# all-reduce-grad-sync — worked example 2: Sync every parameter of a 2-layer model

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `all-reduce-grad-sync`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Grad sync must run over *every* parameter in the model, not just one tensor. The canonical DDP loop is `for p in model.parameters(): all_reduce(p.grad, SUM); p.grad /= world_size`. Each parameter is reduced independently, and the mean rule (divide by world_size) is the same for all of them.

## Worked solution

Here we simulate two ranks each holding their own copy of a `nn.Linear(2, 1)` model with populated grads, and average the grads parameter-by-parameter.

1. Build two model copies (rank 0 and rank 1) and hand each a distinct `.grad` for its weight and bias, mimicking the state right after `loss.backward()` on each rank.
2. For each named parameter we gather the corresponding grad tensor from both ranks. The mock `all_reduce` sums them in place, so both ranks now hold the SUM of that parameter's grad.
3. We divide every grad by `world_size = 2`, converting each SUM into a MEAN.
4. Crucially we iterate over `model.parameters()` so weight AND bias both get synced - forgetting one would let the ranks drift apart. After the loop, rank 0 and rank 1 hold identical grads for every parameter.

The takeaway: the sync loop visits all parameters, and each is reduced+averaged independently with the same world_size divisor.

In [ ]:
import numpy as np
import torch as t
import torch.nn as nn

t.manual_seed(0)
world_size = 2

def make_model_with_grad(scale):
    m = nn.Linear(2, 1)
    for p in m.parameters():
        p.grad = t.ones_like(p) * scale
    return m

rank0 = make_model_with_grad(1.0)   # grads all 1.0
rank1 = make_model_with_grad(3.0)   # grads all 3.0
models = [rank0, rank1]

for params in zip(*[m.parameters() for m in models]):
    total = sum(p.grad.clone() for p in params)
    for p in params:
        p.grad.copy_(total)        # all_reduce SUM (in place)
        p.grad /= world_size       # -> mean

for name, p in rank0.named_parameters():
    print(name, 'synced grad:', p.grad.flatten().tolist())